# PS- 3F: Sign Language MNIST Anomaly Detection

Same dense autoencoder template. Label 24 is the selected anomaly; the other 23 labels are normal.


In [1]:
# Import all packages first
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras import models, layers


In [2]:
# 1. Load updated Sign Language MNIST CSVs. Label 24 is the anomaly.
anomaly_label = 24
train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")
x_normal = train[train["label"] != anomaly_label].drop("label", axis=1).values / 255.0
x_train, x_val = train_test_split(x_normal, test_size=0.1, random_state=42)
x_test = test.drop("label", axis=1).values / 255.0
y_test = (test["label"] == anomaly_label).astype(int)


In [3]:
# 2. Same autoencoder structure for every PS-3 dataset.
encoder = models.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(20, activation="relu"),
    layers.Dense(14, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(4, activation="relu")
])
decoder = models.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(14, activation="relu"),
    layers.Dense(20, activation="relu"),
    layers.Dense(x_train.shape[1], activation="linear")
])
model = models.Sequential([encoder, decoder])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


In [4]:
# 3. Train only on normal data.
model.fit(x_train, x_train, validation_data=(x_val, x_val),
          epochs=20, batch_size=512, shuffle=True, verbose=2)


Epoch 1/20
47/47 - 7s - 146ms/step - loss: 0.2310 - mae: 0.4036 - val_loss: 0.0343 - val_mae: 0.1444
Epoch 2/20
47/47 - 1s - 20ms/step - loss: 0.0259 - mae: 0.1226 - val_loss: 0.0232 - val_mae: 0.1154
Epoch 3/20
47/47 - 1s - 22ms/step - loss: 0.0236 - mae: 0.1166 - val_loss: 0.0232 - val_mae: 0.1156
Epoch 4/20
47/47 - 1s - 21ms/step - loss: 0.0236 - mae: 0.1165 - val_loss: 0.0232 - val_mae: 0.1154
Epoch 5/20
47/47 - 1s - 22ms/step - loss: 0.0236 - mae: 0.1165 - val_loss: 0.0232 - val_mae: 0.1155
Epoch 6/20
47/47 - 1s - 20ms/step - loss: 0.0236 - mae: 0.1164 - val_loss: 0.0232 - val_mae: 0.1155
Epoch 7/20
47/47 - 1s - 21ms/step - loss: 0.0236 - mae: 0.1164 - val_loss: 0.0231 - val_mae: 0.1153
Epoch 8/20
47/47 - 1s - 20ms/step - loss: 0.0236 - mae: 0.1164 - val_loss: 0.0231 - val_mae: 0.1155
Epoch 9/20
47/47 - 1s - 20ms/step - loss: 0.0236 - mae: 0.1164 - val_loss: 0.0231 - val_mae: 0.1153
Epoch 10/20
47/47 - 1s - 21ms/step - loss: 0.0236 - mae: 0.1163 - val_loss: 0.0231 - val_mae: 0.115

In [5]:
# 4. High reconstruction error means label-24 anomaly.
threshold = np.percentile(np.mean((x_val - model.predict(x_val, verbose=0)) ** 2, axis=1), 99)
error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=1)
prediction = (error > threshold).astype(int)
print("Threshold:", threshold)
print(classification_report(y_test, prediction, target_names=["Normal", "Label 24"], zero_division=0))

print("Number of anomalies detected:", prediction.sum())


Threshold: 0.06507036058924849
              precision    recall  f1-score   support

      Normal       0.95      0.99      0.97      6840
    Label 24       0.10      0.02      0.03       332

    accuracy                           0.95      7172
   macro avg       0.53      0.51      0.50      7172
weighted avg       0.91      0.95      0.93      7172

